# News Data Exploration

This notebook loads and explores financial news datasets for sentiment analysis and NLP tasks.

**Dataset:**
- `MAG7_news.csv` - Financial news articles with multiple summary types

## 1. Setup and Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 100)

print("Libraries loaded successfully!")

Libraries loaded successfully!


## 2. Load News Dataset

In [2]:
NEWS_DIR = Path("../data/raw/news")

print("Available news files:")
for f in NEWS_DIR.iterdir():
    if f.is_file() and not f.name.startswith('.'):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name} ({size_mb:.2f} MB)")

Available news files:
  All_external.csv (5465.89 MB)
  MAG7_news.csv (1.01 MB)
  Headlines_by_ticker_date.csv (0.67 MB)
  Filtered_external.csv (1.89 MB)


### 2.1 Load MAG7_news.csv

In [3]:
mag7_external_path = NEWS_DIR / "Filtered_external.csv"

if mag7_external_path.exists():
    df_all_news = pd.read_csv(mag7_external_path)
    print(f"MAG7_news.csv loaded successfully!")
    print(f"   Shape: {df_all_news.shape}")
    print(f"   Columns: {list(df_all_news.columns)}")
else:
    print(f"File not found: {mag7_external_path}")

MAG7_news.csv loaded successfully!
   Shape: (7526, 12)
   Columns: ['Unnamed: 0', 'Date', 'Article_title', 'Stock_symbol', 'Url', 'Publisher', 'Author', 'Article', 'Lsa_summary', 'Luhn_summary', 'Textrank_summary', 'Lexrank_summary']


In [4]:
# Display first few rows
print("First 5 rows:")
df_all_news.head()

First 5 rows:


,Unnamed: 0,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary
0,6680,2020-06-10 07:33:26+00:00,Tech Stocks And FAANGS Strong Again To Start Day As Market Awaits Fed,AAPL,https://www.benzinga.com/government/20/06/16223418/tech-stocks-and-faangs-strong-again-to-start-...,JJ Kinahan,NaN,NaN,NaN,NaN,NaN,NaN
1,6681,2020-06-10 04:14:08+00:00,10 Biggest Price Target Changes For Wednesday,AAPL,https://www.benzinga.com/analyst-ratings/price-target/20/06/16220539/10-biggest-price-target-cha...,Lisa Levin,NaN,NaN,NaN,NaN,NaN,NaN
2,6682,2020-06-10 03:53:47+00:00,"Benzinga Pro's Top 5 Stocks To Watch For Wed., Jun. 10, 2020: AAPL, BAC, NIO, SONO, GLW",AAPL,https://www.benzinga.com/short-sellers/20/06/16220099/benzinga-pros-top-5-stocks-to-watch-for-we...,Benzinga Newsdesk,NaN,NaN,NaN,NaN,NaN,NaN
3,6683,2020-06-10 03:19:25+00:00,"Deutsche Bank Maintains Buy on Apple, Raises Price Target to $350",AAPL,https://www.benzinga.com/news/20/06/16219873/deutsche-bank-maintains-buy-on-apple-raises-price-t...,Benzinga Newsdesk,NaN,NaN,NaN,NaN,NaN,NaN
4,6684,2020-06-10 02:27:11+00:00,"Apple To Let Users Trade In Their Mac Computers For Credit At US, Canada Stores: Report",AAPL,https://www.benzinga.com/news/20/06/16218697/apple-to-let-users-turn-in-their-mac-computers-for-...,Neer Varshney,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Data types
print("Data Types:")
print(df_all_news.dtypes)
print()
print(f"Memory Usage: {df_all_news.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Data Types:
Unnamed: 0            int64
Date                    str
Article_title           str
Stock_symbol            str
Url                     str
Publisher               str
Author              float64
Article             float64
Lsa_summary         float64
Luhn_summary        float64
Textrank_summary    float64
Lexrank_summary     float64
dtype: object

Memory Usage: 2.44 MB


In [6]:
# Missing values
print("Missing values:")
print(df_all_news.isnull().sum())
print()
df_all_news.info()

Missing values:
Unnamed: 0             0
Date                   0
Article_title          0
Stock_symbol           0
Url                    0
Publisher              0
Author              7526
Article             7526
Lsa_summary         7526
Luhn_summary        7526
Textrank_summary    7526
Lexrank_summary     7526
dtype: int64

<class 'pandas.DataFrame'>
RangeIndex: 7526 entries, 0 to 7525
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        7526 non-null   int64  
 1   Date              7526 non-null   str    
 2   Article_title     7526 non-null   str    
 3   Stock_symbol      7526 non-null   str    
 4   Url               7526 non-null   str    
 5   Publisher         7526 non-null   str    
 6   Author            0 non-null      float64
 7   Article           0 non-null      float64
 8   Lsa_summary       0 non-null      float64
 9   Luhn_summary      0 non-null      float64
 10  Textr

## 3. Explore Dataset

In [7]:
if 'Date' in df_all_news.columns:
    df_all_news['Date'] = pd.to_datetime(df_all_news['Date'], utc=True)
    print(f"Date Range: {df_all_news['Date'].min()} to {df_all_news['Date'].max()}")

print(f"Total Articles: {len(df_all_news):,}")

if 'Stock_symbol' in df_all_news.columns:
    print(f"Unique Stock Symbols: {df_all_news['Stock_symbol'].nunique()}")

if 'Publisher' in df_all_news.columns:
    print(f"Unique Publishers: {df_all_news['Publisher'].nunique()}")

Date Range: 2011-03-03 00:00:00+00:00 to 2020-06-10 13:02:47+00:00
Total Articles: 7,526
Unique Stock Symbols: 5
Unique Publishers: 195


In [8]:
# Top stock symbols by article count
if 'Stock_symbol' in df_all_news.columns:
    print("Top 20 Stock Symbols by Article Count:")
    top_symbols = df_all_news['Stock_symbol'].value_counts().head(20)
    print(top_symbols.to_string())

Top 20 Stock Symbols by Article Count:
Stock_symbol
NVDA     3146
TSLA     1875
GOOGL    1754
AAPL      473
AMZN      278


In [9]:
# Top publishers
if 'Publisher' in df_all_news.columns:
    print("Top 10 Publishers:")
    top_publishers = df_all_news['Publisher'].value_counts().head(10)
    print(top_publishers.to_string())

Top 10 Publishers:
Publisher
Benzinga Newsdesk    1671
Lisa Levin            659
Charles Gross         528
Benzinga_Newsdesk     362
Wayne Duggan          304
Paul Quintaro         293
JJ Kinahan            266
Jayson Derrick        243
Neer Varshney         237
Shanthi Rexaline      199


In [10]:
# Check article content availability
if 'Article' in df_all_news.columns:
    article_str = df_all_news['Article'].astype(str)
    has_article = (df_all_news['Article'].notna()) & (article_str != 'N/A') & (article_str.str.strip() != '') & (article_str.str.strip().str.lower() != 'nan')
    print(f"Articles with full text: {has_article.sum():,} ({has_article.mean()*100:.1f}%)")

# Check summary availability
for summary_col in ['Lsa_summary', 'Luhn_summary', 'Textrank_summary', 'Lexrank_summary']:
    if summary_col in df_all_news.columns:
        summary_str = df_all_news[summary_col].astype(str)
        has_summary = (df_all_news[summary_col].notna()) & (summary_str != 'N/A') & (summary_str.str.strip() != '') & (summary_str.str.strip().str.lower() != 'nan')
        print(f"{summary_col}: {has_summary.sum():,} available ({has_summary.mean()*100:.1f}%)")

Articles with full text: 0 (0.0%)
Lsa_summary: 0 available (0.0%)
Luhn_summary: 0 available (0.0%)
Textrank_summary: 0 available (0.0%)
Lexrank_summary: 0 available (0.0%)


## 4. Filter News for Target Stocks

In [11]:
TARGET_TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "NVDA", "META"]

# Filter for our target stocks
if 'Stock_symbol' in df_all_news.columns:
    df_target = df_all_news[df_all_news['Stock_symbol'].isin(TARGET_TICKERS)].copy()
    print(f"Articles for target tickers: {len(df_target):,} out of {len(df_all_news):,} total")
    print()
    print("Articles per target ticker:")
    print(df_target['Stock_symbol'].value_counts().to_string())

Articles for target tickers: 7,526 out of 7,526 total

Articles per target ticker:
Stock_symbol
NVDA     3146
TSLA     1875
GOOGL    1754
AAPL      473
AMZN      278


In [12]:
# Date distribution for target stocks
if 'Date' in df_target.columns:
    print(f"Date Range for target tickers: {df_target['Date'].min()} to {df_target['Date'].max()}")
    print()
    print("Articles per year:")
    print(df_target['Date'].dt.year.value_counts().sort_index().to_string())

Date Range for target tickers: 2011-03-03 00:00:00+00:00 to 2020-06-10 13:02:47+00:00

Articles per year:
Date
2011     226
2012     183
2013     144
2014     118
2015     180
2016     337
2017     615
2018     824
2019    2008
2020    2891


In [ ]:
filtered_external_path = NEWS_DIR / "Filtered_external.csv"
df_target.to_csv(filtered_external_path)

## 4.1 Group Headlines by Ticker and Date

In [ ]:
# Group all headlines for each (Stock_symbol, Date) into a single record.
# This aggregates multiple articles published for the same stock on the same day,
# which is useful for aligning news with daily price data later.

if {'Stock_symbol', 'Date', 'Article_title'}.issubset(df_target.columns):
    grouped = df_target.copy()

    # Normalize Date to a day-level value (drop the time component)
    grouped['Date_only'] = pd.to_datetime(grouped['Date'], utc=True).dt.date

    # Keep only valid headlines
    grouped['Article_title'] = grouped['Article_title'].astype(str)
    grouped = grouped[
        (grouped['Article_title'].str.strip() != '') &
        (grouped['Article_title'].str.strip().str.lower() != 'nan') &
        (grouped['Article_title'] != 'N/A')
    ]

    # Aggregate headlines per ticker per day
    headlines_by_ticker_date = (
        grouped.groupby(['Stock_symbol', 'Date_only'])['Article_title']
        .agg(list)
        .reset_index()
        .rename(columns={'Article_title': 'Headlines', 'Date_only': 'Date'})
    )
    headlines_by_ticker_date['Headline_count'] = headlines_by_ticker_date['Headlines'].apply(len)

    headlines_by_ticker_date = headlines_by_ticker_date.sort_values(
        ['Stock_symbol', 'Date']
    ).reset_index(drop=True)

    print(f"Grouped into {len(headlines_by_ticker_date):,} (ticker, date) groups")
    print()
    print("Total headlines per ticker:")
    print(headlines_by_ticker_date.groupby('Stock_symbol')['Headline_count'].sum().to_string())
    print()
    print("Sample grouped rows:")
    display(headlines_by_ticker_date.head(10))

Grouped into 2,078 (ticker, date) groups

Total headlines per ticker:
Stock_symbol
AAPL      473
AMZN      278
GOOGL    1754
NVDA     3146
TSLA     1875

Sample grouped rows:


,Stock_symbol,Date,Headlines,Headline_count
0,AAPL,2020-03-09,"[Crude Awakening: Energy Sector Takes A 20% Spill As Crude Price War Sends Oil To 4-Year Low, In...",3
1,AAPL,2020-03-10,[Peloton Shares Tick To Session Low As Hearing Report Apple Working On 'Guided Workout' Fitness ...,8
2,AAPL,2020-03-11,"[CME To Shut Down Chicago Trading Floor Over Coronavirus Concerns, Here's How Large Option Trade...",14
3,AAPL,2020-03-12,"[Selling Picks Up In Worst Day Since 1987 As Fear Continues To Drive Markets Lower, Financial Pr...",5
4,AAPL,2020-03-13,"[Canopy Growth's Storz & Bickel Bypasses Apple's Vaping App Ban With New Web App, 88 Stocks Movi...",11
5,AAPL,2020-03-14,"[Barron's Picks And Pans: Roundtable Picks, Airline And Oil Stocks, And More, All Apple Stores O...",2
6,AAPL,2020-03-15,"[Starbucks Switches To 'To Go' Model As Coronavirus Spreads, Closes Some Stores, Loup Ventures' ...",4
7,AAPL,2020-03-16,[McDonald's Switches To 'Walk-In-Take-Out' Model In All Company-Owned Restaurants Due To Coronav...,11
8,AAPL,2020-03-17,"[UPDATE: Apple Previously Announced Closing Of Retail Locations Til Mar. 27, Now Says 'Until Fur...",12
9,AAPL,2020-03-18,"[Oil Giants Persist In Paradoxical Behavior, But They Can Afford It, Travel, Airline Stocks In F...",4


In [ ]:
# Save the grouped headlines to CSV
if 'headlines_by_ticker_date' in dir():
    grouped_path = NEWS_DIR / "Headlines_by_ticker_date.csv"
    headlines_by_ticker_date.to_csv(grouped_path, index=False)
    print(f"Saved grouped headlines to: {grouped_path}")

Saved grouped headlines to: ../data/raw/news/Headlines_by_ticker_date.csv


## 5. Sample Article Content

In [ ]:
# Display a sample article
print("SAMPLE ARTICLE")
print("=" * 60)
sample = df_all_news.iloc[0]
for col in df_all_news.columns:
    val = str(sample[col])[:200]
    print(f"{col}: {val}")
    print()

SAMPLE ARTICLE
Unnamed: 0: 6680

Date: 2020-06-10 07:33:26+00:00

Article_title: Tech Stocks And FAANGS Strong Again To Start Day As Market Awaits Fed

Stock_symbol: AAPL

Url: https://www.benzinga.com/government/20/06/16223418/tech-stocks-and-faangs-strong-again-to-start-day-as-market-awaits-fed

Publisher: JJ Kinahan

Author: nan

Article: nan

Lsa_summary: nan

Luhn_summary: nan

Textrank_summary: nan

Lexrank_summary: nan



In [1]:
# Display one representative record for each stock symbol in a readable format.
# This helps sanity-check the data quality and structure across all tickers.

def display_sample_records(df, group_col='Stock_symbol', max_field_len=200):
    """Print one sample record per group in a human-readable layout.

    Parameters
    ----------
    df : pd.DataFrame
        Source dataframe to sample from.
    group_col : str
        Column used to group records (e.g. the stock symbol).
    max_field_len : int
        Maximum number of characters to display per field (truncates long text).
    """
    if group_col not in df.columns:
        print(f"Column '{group_col}' not found in dataframe.")
        return

    # Take the first record per symbol in a single vectorized pass
    samples = df.groupby(group_col, sort=True).first().reset_index()

    for _, record in samples.iterrows():
        symbol = record[group_col]
        print("=" * 70)
        print(f"SAMPLE ARTICLE \u2014 {symbol}")
        print("=" * 70)
        for col in df.columns:
            value = str(record[col])
            if len(value) > max_field_len:
                value = value[:max_field_len] + "\u2026"
            print(f"{col:>20}: {value}")
        print()


display_sample_records(headlines_by_ticker_date)

NameError: name 'headlines_by_ticker_date' is not defined

In [2]:
headlines_by_ticker_date.head()

NameError: name 'headlines_by_ticker_date' is not defined